# Listener Prior — Seq2Seq Keyphrase Generator (3 datasets, Option A) + fastT5 export

Trains **Flan‑T5‑Small** to generate **keyterms** and **keywords** for the **next USER utterance** only (USER-turn prediction for Deepgram STT injection).
Selects the best checkpoint by **VAL Recall@20 keyterms** and reports **both**:

- Recall@20 keyterms
- Recall@20 keywords

Saves to Drive + exports quantized ONNX via fastT5.


In [ ]:
# ---------------------------
# CONFIG
# ---------------------------
REPO_URL = "https://github.com/ebilal/fSTT.git"
PROJECT_DIR = "/content/listener-prior"

BASE_MODEL = "google/flan-t5-small"  # pretrained; we fine-tune
HISTORY_TURNS = 8
MAX_KEYWORDS = 30
MAX_KEYTERMS = 30

EPOCHS = 3
BATCH_SIZE = 16
LR = 1e-4
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPS = 1e-8

MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 64

TOPK = 20
SELECTION_FIELD = "val_recall@20_keyterms"

# Each run gets a unique timestamped folder
from datetime import datetime
RUN_TAG = "seq2seq_flan_t5_small_user_only"
RUN_NAME = f"{RUN_TAG}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
DRIVE_ROOT = "/content/drive/MyDrive/listener_prior_runs"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
BEST_DIR = f"{RUN_DIR}/best_model"
FASTT5_DIR = f"{RUN_DIR}/best_fastT5_onnx"
print(f"Run artifacts will be saved to: {RUN_DIR}")

RESTAURANT_CSV = "examples/multi_restaurant_phone_orders_5000.csv"


## Mount Drive + set HF_TOKEN from Colab Secrets


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

for k in ['HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
    os.environ.pop(k, None)

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token set from Colab Secrets.')
else:
    print('No HF_TOKEN secret found.')


## Clone repo + install deps


In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"

!python -m pip install -U pip
!grep -v '^torch' requirements.txt > /tmp/requirements_no_torch.txt
!python -m pip install -r /tmp/requirements_no_torch.txt --upgrade

!python -m pip install -U transformers accelerate datasets sentencepiece sacrebleu
!python -m pip install -U fastt5 onnx onnxruntime

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())


## Load conversations (public + restaurant CSV)


In [ ]:
import os, json
import pandas as pd
from typing import List, Dict, Any

def _normalize_role(role: str) -> str:
    role = (role or "").strip().lower()
    if role in {"user", "customer", "human", "client", "guest"}:
        return "USER"
    if role in {"system", "assistant", "agent", "bot", "server"}:
        return "SYSTEM"
    return "SYSTEM"

def load_public_conversations() -> List[Dict[str, Any]]:
    convos: List[Dict[str, Any]] = []
    try:
        from src import data as data_mod
        for fn_name in ["load_dual_dataset_conversations", "load_public_conversations", "get_dual_dataset"]:
            if hasattr(data_mod, fn_name):
                out = getattr(data_mod, fn_name)()
                if isinstance(out, dict):
                    for _, v in out.items():
                        if isinstance(v, list):
                            convos.extend(v)
                elif isinstance(out, list):
                    convos = out
                if convos:
                    print(f"Loaded public via src.data.{fn_name}: {len(convos)}")
                    return convos
    except Exception as e:
        print("Repo loader not available / failed:", repr(e))

    import datasets

    def _extract_text(turn: Dict) -> str:
        for key in ["text", "utterance", "transcript", "content", "sentence"]:
            if key in turn and isinstance(turn[key], str):
                return turn[key]
        return ""

    def _ensure_system_first(turns: List[Dict[str, str]]) -> List[Dict[str, str]]:
        if not turns:
            return turns
        if turns[0].get("speaker") == "SYSTEM":
            return turns
        swapped = []
        for t in turns:
            role = t.get("speaker")
            if role == "USER":
                role = "SYSTEM"
            elif role == "SYSTEM":
                role = "USER"
            swapped.append({"speaker": role, "text": t.get("text", "")})
        return swapped

    def _extract_turns_from_item(item: Dict) -> List[Dict[str, str]]:
        turns: List[Dict[str, str]] = []

        # DailyDialog (roskoN/dailydialog) style: a single list of utterances.
        # Roles are not explicitly provided; alternate SYSTEM/USER by index.
        if "utterances" in item and isinstance(item["utterances"], list) and item["utterances"]:
            utterances = item["utterances"]
            for i, text in enumerate(utterances):
                if not isinstance(text, str):
                    continue
                text = text.strip()
                if not text:
                    continue
                role = "SYSTEM" if i % 2 == 0 else "USER"
                turns.append({"speaker": role, "text": text})
            if turns:
                return turns

        if "turns" in item and isinstance(item["turns"], dict):
            speakers = item["turns"].get("speaker") or []
            utterances = item["turns"].get("utterance") or []
            for speaker, text in zip(speakers, utterances):
                role = "USER" if str(speaker) == "0" else "SYSTEM"
                text = (text or "").strip()
                if text:
                    turns.append({"speaker": role, "text": text})
            return turns

        turns_list = None
        for key in ["turns", "dialogue", "dialog", "utterances", "messages"]:
            if key in item and isinstance(item[key], list):
                turns_list = item[key]
                break
        if turns_list is None:
            return []

        for idx, turn in enumerate(turns_list):
            if isinstance(turn, str):
                role = "SYSTEM" if idx % 2 == 0 else "USER"
                text = turn
            elif isinstance(turn, dict):
                role = _normalize_role(turn.get("speaker") or turn.get("role") or turn.get("participant") or "")
                text = _extract_text(turn)
                if not text and "utterances" in turn and isinstance(turn["utterances"], str):
                    text = turn["utterances"]
            else:
                continue
            text = (text or "").strip()
            if text:
                turns.append({"speaker": role, "text": text})
        return turns

    DATASET_SOURCES = [
        ("dailydialog", ["roskoN/dailydialog", "daily_dialog"]),
        ("multiwoz", ["pfb30/multi_woz_v22"]),
    ]

    token = (
        os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGINGFACE_HUB_TOKEN")
        or os.environ.get("HUGGING_FACE_HUB_TOKEN")
        or None
    )

    def _load(repo_id: str):
        """Try standard load; fall back to auto-converted Parquet branch."""
        kwargs = {"token": token} if token else {}
        # Strategy 1: standard load (works if the dataset is already Parquet/JSON/CSV)
        try:
            return datasets.load_dataset(repo_id, split="train", **kwargs)
        except Exception as e1:
            err = str(e1).lower()
            if "script" not in err and "trust_remote_code" not in err:
                raise
            print(f"  {repo_id}: script-based dataset, trying Parquet branch …")
        # Strategy 2: load from the auto-converted refs/convert/parquet branch
        return datasets.load_dataset(
            repo_id, split="train",
            revision="refs/convert/parquet", **kwargs,
        )

    print("Using HF fallback datasets:", DATASET_SOURCES)

    for ds_name, repo_ids in DATASET_SOURCES:
        dataset = None
        used_repo = None
        for repo_id in repo_ids:
            try:
                dataset = _load(repo_id)
                used_repo = repo_id
                break
            except Exception as e:
                print(f"Skipping {repo_id} due to load error: {repr(e)}")
                continue
        if dataset is None:
            continue
        if ds_name == "dailydialog" and used_repo == "daily_dialog":
            print("Warning: using legacy 'daily_dialog' source; prefer 'roskoN/dailydialog'.")

        for i, item in enumerate(dataset):
            turns = _extract_turns_from_item(item)
            if not turns:
                dialog = item.get("dialog")
                if isinstance(dialog, list):
                    turns = [{"speaker": "SYSTEM" if j % 2 == 0 else "USER", "text": str(t)} for j, t in enumerate(dialog) if isinstance(t, str)]
            turns = _ensure_system_first(turns)
            if len(turns) >= 2:
                convos.append({"dialog_id": f"{used_repo}:train:{i}", "turns": turns})

    print("Loaded HF public conversations:", len(convos))
    return convos

def load_restaurant_csv(path: str) -> List[Dict[str, Any]]:
    assert os.path.exists(path), f"Missing CSV at {path}."
    df = pd.read_csv(path)
    required = {"dialog_id", "utterance_id", "speaker", "text"}
    assert required.issubset(set(df.columns)), f"CSV must include {required}, got {set(df.columns)}"
    convos: List[Dict[str, Any]] = []
    for did, g in df.sort_values(["dialog_id", "utterance_id"]).groupby("dialog_id"):
        turns = [
            {"speaker": _normalize_role(str(r["speaker"])), "text": str(r["text"])}
            for _, r in g.iterrows()
        ]
        if len(turns) >= 2:
            convos.append({"dialog_id": f"restaurant:{did}", "turns": turns})
    print("Loaded restaurant conversations:", len(convos))
    return convos

public_convos = load_public_conversations()
restaurant_convos = load_restaurant_csv(RESTAURANT_CSV)
all_convos = public_convos + restaurant_convos
print("TOTAL conversations:", len(all_convos))


In [ ]:
# Sanity check: show a few dialogs from each source
from itertools import islice

def _print_samples(convos, label, n=2):
    print("\n" + "="*80)
    print(f"{label} (showing {n})")
    print("="*80)
    for c in list(islice(convos, n)):
        print("dialog_id:", c["dialog_id"])
        for t in c["turns"][:3]:
            print(f"  {t['speaker']}: {t['text']}")
        if len(c["turns"]) > 3:
            print("  ...")

public_only = [c for c in all_convos if not c["dialog_id"].startswith("restaurant:")]
restaurant_only = [c for c in all_convos if c["dialog_id"].startswith("restaurant:")]

_print_samples(public_only, "Public datasets")
_print_samples(restaurant_only, "Restaurant CSV")


## Build Option A examples + global candidate pool


In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def build_examples(convos, history_turns: int):
    # --- Pass 1: collect all USER-turn targets + history strings ---
    raw = []
    for c in convos:
        turns = c["turns"]
        for t in range(1, len(turns)):
            target = turns[t]
            # Only predict USER utterances (the ones we need Deepgram hints for)
            if target["speaker"] != "USER":
                continue
            hist = turns[max(0, t - history_turns):t]
            inp = "\n".join([f'{h["speaker"]}: {h["text"]}' for h in hist]).strip()
            raw.append({
                "dialog_id": c["dialog_id"],
                "turn_idx": t,
                "input_text": inp,
                "target_text_raw": target["text"],
            })
    if not raw:
        return []

    all_texts = [r["target_text_raw"] for r in raw]
    print(f"Collected {len(all_texts)} USER-turn targets. Running batched TF-IDF …")

    # --- Batch TF-IDF: one vectorizer per ngram range, over the whole corpus ---
    def batch_top_terms(texts, ngram_range, max_items):
        try:
            vec = TfidfVectorizer(stop_words="english", ngram_range=ngram_range, min_df=1)
            tfidf = vec.fit_transform(texts).tocsr()
        except ValueError:
            return [[] for _ in texts]
        if tfidf.shape[1] == 0:
            return [[] for _ in texts]
        names = vec.get_feature_names_out()
        results = []
        for i in range(tfidf.shape[0]):
            row = tfidf[i]
            if row.nnz == 0:
                results.append([])
                continue
            cols = row.indices
            top = np.argsort(-row.data)[:max_items]
            results.append([names[cols[j]] for j in top])
        return results

    all_keywords = batch_top_terms(all_texts, (1, 1), MAX_KEYWORDS)
    print("  keywords (unigrams) done")
    all_keyterms = batch_top_terms(all_texts, (2, 3), MAX_KEYTERMS)
    print("  keyterms (2-3 grams) done")

    # --- Regex patterns for numbers / times / dates (matches src/prior.py) ---
    _NUMERIC_RE = re.compile(r"\b\d{1,4}(?::\d{2})?\b")
    _TIME_RE    = re.compile(r"\b\d{1,2}(?:am|pm)\b", re.IGNORECASE)
    _DATE_RE    = re.compile(r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\w*\b", re.IGNORECASE)

    def _dedupe(items):
        seen = set(); out = []
        for x in items:
            k = x.lower()
            if k not in seen:
                seen.add(k); out.append(x)
        return out

    # --- Pass 2: assemble examples with pre-computed terms ---
    exs = []
    for i, r in enumerate(raw):
        txt = r["target_text_raw"]
        numeric = _NUMERIC_RE.findall(txt) + _TIME_RE.findall(txt) + _DATE_RE.findall(txt)
        keywords = _dedupe(all_keywords[i] + numeric)[:MAX_KEYWORDS]
        keyterms = _dedupe(all_keyterms[i])[:MAX_KEYTERMS]

        tgt = "keyterms: " + "; ".join(keyterms) + "\n"
        tgt += "keywords: " + "; ".join(keywords)

        exs.append({
            "dialog_id": r["dialog_id"],
            "turn_idx": r["turn_idx"],
            "input_text": r["input_text"],
            "target_text": tgt,
        })
    return exs

examples = build_examples(all_convos, HISTORY_TURNS)
candidates = sorted(set(e["target_text"] for e in examples if e["target_text"]))
print("examples:", len(examples), "unique candidates:", len(candidates))


## Split train/val/test by dialog id

In [ ]:
from collections import defaultdict
import random

rng = random.Random(7)
by_dialog = defaultdict(list)
for e in examples:
    by_dialog[e["dialog_id"]].append(e)

dialog_ids = list(by_dialog.keys())
rng.shuffle(dialog_ids)

n = len(dialog_ids)
n_train = int(0.8*n)
n_val = int(0.1*n)

train_ids = set(dialog_ids[:n_train])
val_ids   = set(dialog_ids[n_train:n_train+n_val])
test_ids  = set(dialog_ids[n_train+n_val:])

train_ex = [e for did in train_ids for e in by_dialog[did]]
val_ex   = [e for did in val_ids for e in by_dialog[did]]
test_ex  = [e for did in test_ids for e in by_dialog[did]]

print("dialogs:", n, "train/val/test:", len(train_ids), len(val_ids), len(test_ids))
print("examples:", len(train_ex), len(val_ex), len(test_ex))

## Recall@20 split: keyterms vs keywords


In [ ]:
def parse_keyterms_keywords(text: str):
    keyterms=[]
    keywords=[]
    for line in text.splitlines():
        line=line.strip()
        if line.lower().startswith("keyterms:"):
            rhs=line.split(":",1)[1]
            keyterms=[t.strip() for t in rhs.split(";") if t.strip()]
        elif line.lower().startswith("keywords:"):
            rhs=line.split(":",1)[1]
            keywords=[t.strip() for t in rhs.split(";") if t.strip()]
    def dedup(xs):
        seen=set(); out=[]
        for x in xs:
            if x not in seen:
                seen.add(x); out.append(x)
        return out
    return dedup(keyterms), dedup(keywords)

def recall_at_k(gt, pred, k=20):
    gt_set=set(gt)
    if not gt_set:
        return 0.0
    return len(set(pred[:k]) & gt_set) / len(gt_set)

def split_recalls(gt_text: str, pred_text: str, k=20):
    gt_t, gt_w = parse_keyterms_keywords(gt_text)
    pr_t, pr_w = parse_keyterms_keywords(pred_text)
    return {
        "recall@20_keyterms": recall_at_k(gt_t, pr_t, k),
        "recall@20_keywords": recall_at_k(gt_w, pr_w, k),
    }


## Train Flan‑T5 Small + select best by VAL Recall@20 keyterms


In [ ]:
import os, json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.auto import tqdm

os.makedirs(RUN_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SEQ2SEQ_PROMPT = "predict keyterms for next user utterance\n"

class PairDS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    inputs=[SEQ2SEQ_PROMPT + b["input_text"] for b in batch]
    targets=[b["target_text"] for b in batch]
    enc=tokenizer(inputs, max_length=MAX_INPUT_TOKENS, truncation=True, padding=True, return_tensors="pt")
    lab=tokenizer(targets, max_length=128, truncation=True, padding=True, return_tensors="pt")["input_ids"]
    lab[lab==tokenizer.pad_token_id] = -100
    enc["labels"]=lab
    return enc

train_dl=DataLoader(PairDS(train_ex), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
opt=AdamW(model.parameters(), lr=LR, betas=(ADAM_BETA1, ADAM_BETA2), eps=ADAM_EPS, weight_decay=WEIGHT_DECAY)
total_steps = len(train_dl) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(opt, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

@torch.no_grad()
def eval_recalls(m, rows, max_batches=80):
    m.eval()
    sum_t=0.0; sum_w=0.0; n=0
    dl = DataLoader(PairDS(rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    for bi, batch in enumerate(dl):
        if bi >= max_batches:
            break
        inputs={k:v.to(device) for k,v in batch.items() if k!="labels"}
        gen=m.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1)
        pred_txt = tokenizer.batch_decode(gen, skip_special_tokens=True)
        start=bi*BATCH_SIZE
        gts=[r["target_text"] for r in rows[start:start+len(pred_txt)]]
        for g,p in zip(gts, pred_txt):
            m2 = split_recalls(g,p,k=TOPK)
            sum_t += m2["recall@20_keyterms"]
            sum_w += m2["recall@20_keywords"]
            n += 1
    return {"recall@20_keyterms": float(sum_t/max(1,n)), "recall@20_keywords": float(sum_w/max(1,n))}

best={SELECTION_FIELD:-1.0,"epoch":None}
history=[]

for epoch in range(1, EPOCHS+1):
    model.train()
    pbar=tqdm(train_dl, desc=f"epoch {epoch}/{EPOCHS}")
    total=0.0
    for batch in pbar:
        batch={k:v.to(device) for k,v in batch.items()}
        out=model(**batch)
        loss=out.loss
        loss.backward()
        opt.step()
        scheduler.step()
        opt.zero_grad(set_to_none=True)
        total += loss.item()
        pbar.set_postfix(loss=total/(pbar.n+1))

    val_m = eval_recalls(model, val_ex, max_batches=80)
    print(f"VAL Recall@20 keyterms: {val_m['recall@20_keyterms']:.6f} | keywords: {val_m['recall@20_keywords']:.6f}")
    history.append({"epoch":epoch, "val_recall@20_keyterms": val_m["recall@20_keyterms"], "val_recall@20_keywords": val_m["recall@20_keywords"]})

    if val_m["recall@20_keyterms"] > best[SELECTION_FIELD]:
        best={SELECTION_FIELD: val_m["recall@20_keyterms"], "epoch": epoch, **val_m}
        os.makedirs(BEST_DIR, exist_ok=True)
        model.save_pretrained(BEST_DIR)
        tokenizer.save_pretrained(BEST_DIR)
        with open(os.path.join(RUN_DIR,"best_metrics.json"),"w") as f:
            json.dump({"selection_metric": SELECTION_FIELD, **best, "history": history}, f, indent=2)
        print("✓ Saved BEST model:", BEST_DIR)

print("BEST:", best)


## Test evaluation + save performance.json


In [ ]:
import os, json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

best_tok = AutoTokenizer.from_pretrained(BEST_DIR)
best_model = AutoModelForSeq2SeqLM.from_pretrained(BEST_DIR).to(device)

@torch.no_grad()
def eval_test(m, rows, max_batches=200):
    m.eval()
    sum_t=0.0; sum_w=0.0; n=0
    dl = DataLoader(PairDS(rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    for bi, batch in enumerate(dl):
        if bi >= max_batches:
            break
        inputs={k:v.to(device) for k,v in batch.items() if k!="labels"}
        gen=m.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1)
        pred_txt = best_tok.batch_decode(gen, skip_special_tokens=True)
        start=bi*BATCH_SIZE
        gts=[r["target_text"] for r in rows[start:start+len(pred_txt)]]
        for g,p in zip(gts, pred_txt):
            m2=split_recalls(g,p,k=TOPK)
            sum_t += m2["recall@20_keyterms"]
            sum_w += m2["recall@20_keywords"]
            n += 1
    return {"recall@20_keyterms": float(sum_t/max(1,n)), "recall@20_keywords": float(sum_w/max(1,n))}

test_m = eval_test(best_model, test_ex, max_batches=200)
print(f"TEST Recall@20 keyterms: {test_m['recall@20_keyterms']:.6f} | keywords: {test_m['recall@20_keywords']:.6f}")

perf={
    "run_name": RUN_NAME,
    "base_model": BASE_MODEL,
    "history_turns": HISTORY_TURNS,
    "selection_metric": SELECTION_FIELD,
    "best_epoch": best.get("epoch"),
    "best_val_recall@20_keyterms": best.get("recall@20_keyterms"),
    "best_val_recall@20_keywords": best.get("recall@20_keywords"),
    "test_recall@20_keyterms": test_m["recall@20_keyterms"],
    "test_recall@20_keywords": test_m["recall@20_keywords"],
}
with open(os.path.join(RUN_DIR,"performance.json"),"w") as f:
    json.dump(perf,f,indent=2)
print("Saved:", os.path.join(RUN_DIR,"performance.json"))


## Export BEST to fastT5 (quantized ONNX)


In [ ]:
import os
from fastT5 import export_and_get_onnx_model

os.makedirs(FASTT5_DIR, exist_ok=True)
onnx_model = export_and_get_onnx_model(BEST_DIR, onnx_dir=FASTT5_DIR, quantize=True)
print("Saved fastT5 artifacts:", FASTT5_DIR)


## Example predictions (2 test examples)


In [ ]:
def generate_pred(history_text: str) -> str:
    inp = SEQ2SEQ_PROMPT + history_text
    enc = best_tok([inp], truncation=True, padding=True, max_length=MAX_INPUT_TOKENS, return_tensors="pt").to(device)
    gen = best_model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, num_beams=1)
    return best_tok.batch_decode(gen, skip_special_tokens=True)[0]

for p in test_ex[:2]:
    pred = generate_pred(p["input_text"])
    gt_t, gt_w = parse_keyterms_keywords(p["target_text"])
    pr_t, pr_w = parse_keyterms_keywords(pred)

    ov_t = sorted(set(pr_t[:20]) & set(gt_t))
    ov_w = sorted(set(pr_w[:20]) & set(gt_w))

    print("="*90)
    print("HISTORY:\n", p["input_text"][:800])
    print("\nPRED text:\n", pred)

    print("\nGT keyterms:", gt_t[:30])
    print("PRED keyterms@20:", pr_t[:20])
    print("Overlap keyterms:", ov_t, f"(recall={len(ov_t)}/{len(set(gt_t)) if gt_t else 0})")

    print("\nGT keywords:", gt_w[:30])
    print("PRED keywords@20:", pr_w[:20])
    print("Overlap keywords:", ov_w, f"(recall={len(ov_w)}/{len(set(gt_w)) if gt_w else 0})")


## Production usage (seq2seq)

The generated text contains two lines (`keyterms:` and `keywords:`). In production, parse them separately with `parse_keyterms_keywords(...)`
and take the first 20 of each.

For real-time latency:
- `num_beams=1`
- keep `max_new_tokens` small
- cap input length
- prefer fastT5 ONNX for CPU.
